# A foreign value is refused

The alternative names a different dictionary's FIRST, even though its code is also first.

Run in a fresh kernel using the repository's Python environment. The matching script is `47_error_foreign_value.py`.


## Prepare the model

These declarations supply the complete model needed by this case. Run this cell before the experiment.


In [1]:
from __future__ import annotations

from typing import Annotated

from pydantic import Field

from aoa.action_machine.domain import (
    BaseCoreEntity,
    BaseEnumEntity,
    BaseSpecEntity,
    Classifier,
    Generalization,
    Inverse,
    Rel,
    Specialization,
)
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.entity import entity


class MusicDomain(BaseDomain):
    """Group the music catalogue declarations."""

    name = "music"
    description = "A music catalogue"


@entity(description="Kind of VinylRecordCoreEntity", domain=MusicDomain)
class VinylRecordKindEnumEntity(BaseEnumEntity):
    """Declare the codes the axis borrows."""

    FIRST = VinylRecordKindEnumEntity(code="first", display_name="First")  # noqa: F821
    REPRESS = VinylRecordKindEnumEntity(code="repress", display_name="Repress")  # noqa: F821
    TEST = VinylRecordKindEnumEntity(code="test", display_name="Test")  # noqa: F821


@entity(description="Independent vocabulary", domain=MusicDomain)
class OtherKindEnumEntity(BaseEnumEntity):
    """Reuse a code in a different vocabulary."""

    FIRST = OtherKindEnumEntity(code="first", display_name="Other first")  # noqa: F821


@entity(description="Vinyl record", domain=MusicDomain)
class VinylRecordCoreEntity(BaseCoreEntity):
    """Describe the information shared by all pressings."""

    id: str = Field(description="Record identifier")
    title: str = Field(description="Album title")
    pressing: Annotated[
        Specialization[FirstPressSpecEntity | RepressSpecEntity | TestPressSpecEntity, VinylRecordKindEnumEntity],
        Inverse(field_name="record"),
    ] = Rel(description="Details of this pressing")


@entity(description="First pressing", domain=MusicDomain)
class FirstPressSpecEntity(BaseSpecEntity):
    """Describe the stamper used for a first pressing."""

    id: str = Field(description="The head record identifier")
    stamper: str = Field(description="Stamper code")
    record: Annotated[
        Generalization[VinylRecordCoreEntity],
        Classifier(field="record", variant=OtherKindEnumEntity.FIRST),
        Inverse(VinylRecordCoreEntity, "pressing"),
    ] = Rel(description="Record described by this pressing")


@entity(description="Re-pressing", domain=MusicDomain)
class RepressSpecEntity(BaseSpecEntity):
    """Describe when an album was pressed again."""

    id: str = Field(description="The head record identifier")
    year: int = Field(description="Re-pressing year")
    record: Annotated[
        Generalization[VinylRecordCoreEntity],
        Classifier(field="record", variant=VinylRecordKindEnumEntity.REPRESS),
        Inverse(VinylRecordCoreEntity, "pressing"),
    ] = Rel(description="Record described by this pressing")


@entity(description="Test pressing", domain=MusicDomain)
class TestPressSpecEntity(BaseSpecEntity):
    """Describe who approved a test pressing."""

    id: str = Field(description="The head record identifier")
    approved_by: str = Field(description="Reviewer name")
    record: Annotated[
        Generalization[VinylRecordCoreEntity],
        Classifier(field="record", variant=VinylRecordKindEnumEntity.TEST),
        Inverse(VinylRecordCoreEntity, "pressing"),
    ] = Rel(description="Record described by this pressing")


for model in (VinylRecordCoreEntity, FirstPressSpecEntity, RepressSpecEntity, TestPressSpecEntity):
    model.model_rebuild()


## Run the experiment

The alternative names a different dictionary's FIRST, even though its code is also first.


In [2]:
from aoa.action_machine.domain.exceptions import SpecializationDeclarationError
from aoa.action_machine.graph.validators.entity_specialization_validator import validate_entity_specializations

try:
    validate_entity_specializations(
        [VinylRecordCoreEntity, FirstPressSpecEntity, RepressSpecEntity, TestPressSpecEntity]
    )
except SpecializationDeclarationError as error:
    print(type(error).__name__ + ": " + str(error))
else:
    raise AssertionError("The invalid declaration was accepted")


SpecializationDeclarationError: Specialization 'pressing' on entity 'VinylRecordCoreEntity' is invalid: alternative 'FirstPressSpecEntity' names a value OtherKindEnumEntity(code='first', display_name='Other first') that 'VinylRecordKindEnumEntity' does not declare


## Explain the result

A value belongs to one dictionary. Equal code text does not make values interchangeable; select the attribute of the dictionary named by the head.
